# README
Base template for HACS200 honeypot project data analysis. Subject to change.

### **Instructions**
Please make a copy of this template for your honeypot project. Rename the file to follow this structure:

{2}F\_{Fallacy}\_HACS200\_Data

Additionally, please keep this document as organized as you can. Consider commenting your code and/or providing some sort of documentation. This will make it easier for you, your teammates, and course staff to understand what has been written in the long run!

***
# Team Information
Team Name: #TODO

Team Members: #TODO

***
# Set Up
Importing libraries. Please run this first.

In [159]:
# importing libraries
%pip install scikit-posthocs nbformat


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [160]:
# more library imports
import pandas as pd
import scipy as sp
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as sm
import scikit_posthocs as skph
from scipy.stats import f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import MultiComparison


***
# File Uploads
Upload your data here by running the cell below!

In [161]:
# colab asks for the csv. local runs read sessions_full.csv in this folder
from pathlib import Path

try:
    from google.colab import files
    uploads = files.upload()
    name = next(iter(uploads))
except ModuleNotFoundError:
    name = Path("sessions_full.csv")

df = pd.read_csv(name)
df.head()


,timestamp,honeypot,config,attacker_ip,session_seconds,session_ms,reason,commands,keystrokes,active_ms,command_count,command_text
0,2026-10-28 19:50:56,hp1,1,179.43.153.2,1,1117.0,attacker left,1,0,1117.0,1,"[""/bin/./uname -s -v -n -r -m""]"
1,2026-10-28 19:52:20,hp1,3,179.43.153.2,1,969.0,attacker left,1,0,969.0,1,"[""/bin/./uname -s -v -n -r -m""]"
2,2026-10-28 19:53:42,hp1,2,179.43.153.2,1,971.0,attacker left,1,0,971.0,1,"[""/bin/./uname -s -v -n -r -m""]"
3,2026-10-28 19:55:01,hp1,3,179.43.153.2,1,1359.0,attacker left,1,0,1359.0,1,"[""/bin/./uname -s -v -n -r -m""]"
4,2026-10-28 19:56:21,hp1,5,179.43.153.2,1,968.0,attacker left,1,0,968.0,1,"[""/bin/./uname -s -v -n -r -m""]"


***
# Processing Data
You may find the work you did in HACS101 Mini Project 2 to be helpful for this portion. You are welcome to add additional markdown/code blocks!

## How rows are counted

Each row is one finished attack, built from that attack's `mitm.log` by `export_sessions.py` on the VM.

**Time is active time** (`active_ms`, in milliseconds). It runs from the moment the attacker is inside the container to the end of the session. When the honeypot kicks an attacker for being idle, the idle wait is not counted: active time stops at the attacker's last command or keystroke. Sessions that end any other way use the full session length.

**Commands** (`command_count`) count each chained command on its own. The line is split on `;`, `&&`, `||`, a background `&`, and newlines. A separator inside quotes or escaped with a backslash does not split, and a pipe (`|`) stays one command. Examples: `cd /tmp; ls; echo "done"` = 3, `mkdir mydir && cd mydir && touch file.txt` = 3, `cd /nonexistent || echo "Directory not found"` = 2.

**Time limit.** Attackers are kicked after 300 seconds of wall-clock time. An attack counts as reaching the time limit when its full session reached 300 seconds, or when `reason` is a session timeout.

**Unique IPs** keep one row per address: the earliest attack from that address, in that attack's configuration. An IP whose first attack hit the time limit is left out of the excluding-timeouts plot, and a later attack from that IP is not substituted.


In [162]:
# one row is one finished attack. times come from the mitm log timestamps, in ms
df["timestamp"] = pd.to_datetime(df["timestamp"])
for column in ["session_ms", "active_ms", "command_count"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["session_seconds"] = df["session_ms"] / 1000
df["config"] = df["config"].astype(str).str.replace(r"\.0$", "", regex=True)
# rows with no session_ms disconnected before getting a shell, so there is no session to time
df = df.dropna(subset=["session_ms", "active_ms"])
df = df[df["config"].isin(["1", "2", "3", "4", "5"])].copy()

LABELS = {
    "1": "1 control",
    "2": "2 25% ascii",
    "3": "3 50% ascii",
    "4": "4 75% ascii",
    "5": "5 100% ascii",
}
ORDER = list(LABELS.values())
TIME_LIMIT = 300
TIMEOUT_REASONS = {"session timeout", "kicked at 5 minutes", "timeout"}
SIGNIFICANCE = 0.1
NORMAL_ALPHA = 0.05

df["config_label"] = df["config"].map(LABELS)
df["hit_time_limit"] = (df["session_seconds"] >= TIME_LIMIT) | (
    df["reason"].fillna("").str.strip().str.lower().isin(TIMEOUT_REASONS)
)

# timeouts are the 5 minute kick. idle kicks stay in, with their idle wait removed
kept = df[~df["hit_time_limit"]].copy()

# one row per ip: the earliest attack, kept in that attack's configuration
known = df[df["attacker_ip"].fillna("").ne("") & df["attacker_ip"].ne("unknown")].copy()
first = known.sort_values("timestamp").drop_duplicates("attacker_ip", keep="first")
first_kept = first[~first["hit_time_limit"]].copy()

pd.set_option("display.max_rows", 200)
print(f"data from {df['timestamp'].min()} to {df['timestamp'].max()} (eastern)")
print(f"attacks: {len(df)}")
print(f"unique ips: {df['attacker_ip'].nunique()}")
print(f"hit the 5 minute limit: {int(df['hit_time_limit'].sum())}")
print(f"idle kicks: {int((df['reason'] == 'idle timeout').sum())}")
print(f"attacks excluding the time limit: {len(kept)}")
print(f"first attacks: {len(first)}")
print(f"first attacks excluding the time limit: {len(first_kept)}")
df.head()


data from 2026-10-28 19:50:56 to 2026-11-11 18:30:48 (eastern)
attacks: 45764
unique ips: 1929
hit the 5 minute limit: 0
idle kicks: 36
attacks excluding the time limit: 45764
first attacks: 1929
first attacks excluding the time limit: 1929


,timestamp,honeypot,config,attacker_ip,session_seconds,session_ms,reason,commands,keystrokes,active_ms,command_count,command_text,config_label,hit_time_limit
0,2026-10-28 19:50:56,hp1,1,179.43.153.2,1.117,1117.0,attacker left,1,0,1117.0,1,"[""/bin/./uname -s -v -n -r -m""]",1 control,False
1,2026-10-28 19:52:20,hp1,3,179.43.153.2,0.969,969.0,attacker left,1,0,969.0,1,"[""/bin/./uname -s -v -n -r -m""]",3 50% ascii,False
2,2026-10-28 19:53:42,hp1,2,179.43.153.2,0.971,971.0,attacker left,1,0,971.0,1,"[""/bin/./uname -s -v -n -r -m""]",2 25% ascii,False
3,2026-10-28 19:55:01,hp1,3,179.43.153.2,1.359,1359.0,attacker left,1,0,1359.0,1,"[""/bin/./uname -s -v -n -r -m""]",3 50% ascii,False
4,2026-10-28 19:56:21,hp1,5,179.43.153.2,0.968,968.0,attacker left,1,0,968.0,1,"[""/bin/./uname -s -v -n -r -m""]",5 100% ascii,False


## Number of attacks per IP address

A few addresses with a large share means repeat scanners, rather than a separate person for every row. This checks whether one address dominates, and whether addresses look wrong (blank, `unknown`, or our own host addresses).


In [163]:
ip_counts = df["attacker_ip"].value_counts()
print(f"unique ips: {len(ip_counts)}")
print(f"top ip {ip_counts.index[0]}: {ip_counts.iloc[0]} attacks ({100 * ip_counts.iloc[0] / len(df):.1f}%)")
print(f"top 10 ips: {ip_counts.head(10).sum()} attacks ({100 * ip_counts.head(10).sum() / len(df):.1f}%)")
print(f"ips seen only once: {int((ip_counts == 1).sum())}")
print(f"blank or unknown ips: {int(df['attacker_ip'].fillna('').isin(['', 'unknown']).sum())}")
own = df["attacker_ip"].astype(str).str.match(r"^(10\.|127\.|192\.168\.|128\.8\.238\.)")
print(f"private or honeypot ips: {int(own.sum())}")
display(ip_counts.rename("attacks").to_frame())


unique ips: 1929
top ip 217.60.76.32: 1648 attacks (3.6%)
top 10 ips: 10771 attacks (23.5%)
ips seen only once: 1072
blank or unknown ips: 0
private or honeypot ips: 0


,attacks
attacker_ip,
217.60.76.32,1648
109.160.32.180,1386
109.160.32.156,1097
109.160.32.126,1057
109.160.32.179,1039
...,...
217.182.253.249,1
141.94.26.113,1
106.12.51.76,1


In [164]:
# attacks per ip, top 30 addresses. the rest are summed into one bar
TOP_IPS = 30
top = ip_counts.head(TOP_IPS)
rest = ip_counts.iloc[TOP_IPS:].sum()
bars = pd.concat([top, pd.Series({f"other {len(ip_counts) - TOP_IPS} ips": rest})])
fig_ip_counts = px.bar(
    x=bars.index,
    y=bars.values,
    labels={"x": "attacker ip", "y": "attacks"},
    title=f"number of attacks per ip address (top {TOP_IPS}, everyone else in the last bar)",
)
fig_ip_counts.update_layout(width=1300, height=520, xaxis_tickangle=-60)
fig_ip_counts.show()

# how many ips made 1 attack, 2 attacks, and so on
fig_ip_hist = px.histogram(
    x=ip_counts.values,
    nbins=60,
    log_y=True,
    labels={"x": "attacks from one ip", "y": "ips"},
    title="distribution of attacks per ip address (log scale)",
)
fig_ip_hist.update_layout(width=1300, height=420, yaxis_title="number of ips (log scale)")
fig_ip_hist.show()


**What do you observe?**

The numbers below are from data through Nov 11, 2026, 6:30 pm eastern. They change as new attacks arrive.

- 45,764 attacks came from 1,929 IP addresses. The spread is very uneven: 1,072 addresses (56%) attacked exactly once, while the top 10 addresses made 23.5% of all attacks.
- The single busiest address, 217.60.76.32, made 1,648 attacks (3.6%).
- Counting per address understates how concentrated the traffic is. The /24 block 109.160.32.x (59 addresses) made 29,433 attacks, 64% of the whole dataset. Every one of them sends `uname -s -v -n -r -m` and leaves. This is one operator rotating through a block of addresses, not 59 separate people.
- No IP issues: no attack has a blank or `unknown` IP, and none comes from a private address or our own honeypot addresses. The attacker IP is the address that logged in. That bug was fixed (see the roadblocks section).
- Because one scanner dominates the per-attack data, the unique-IP plots below give a better picture of how different attackers behave.

## Number of attacks per honeypot configuration

Randomization is working when the five configurations each get about 20% of attacks.


In [165]:
config_counts = df["config_label"].value_counts().reindex(ORDER)
config_share = (100 * config_counts / config_counts.sum()).round(1)
summary = pd.DataFrame({"attacks": config_counts, "percent": config_share})
display(summary)
spread = config_share.max() - config_share.min()
print(f"share spread: {spread:.1f} percentage points")

fig_config_counts = px.bar(
    summary.reset_index(),
    x="config_label",
    y="attacks",
    text="percent",
    labels={"config_label": "honeypot configuration", "attacks": "attacks"},
    title="number of attacks per honeypot configuration",
)
fig_config_counts.update_traces(texttemplate="%{text}%", textposition="outside")
fig_config_counts.add_hline(
    y=len(df) / len(ORDER), line_dash="dash", annotation_text="even split (20%)"
)
fig_config_counts.update_layout(width=900, height=480)
fig_config_counts.show()

# same split per honeypot container, since each container is re-randomized on every recycle
display(pd.crosstab(df["honeypot"], df["config_label"]).reindex(columns=ORDER))


,attacks,percent
config_label,,
1 control,9189,20.1
2 25% ascii,9084,19.8
3 50% ascii,9221,20.1
4 75% ascii,9177,20.1
5 100% ascii,9093,19.9


share spread: 0.3 percentage points


config_label,1 control,2 25% ascii,3 50% ascii,4 75% ascii,5 100% ascii
honeypot,,,,,
hp1,2268,2249,2319,2379,2281
hp2,1996,1988,1997,1998,1967
hp3,2707,2605,2688,2556,2611
hp4,2218,2242,2217,2244,2234


**What do you observe?**

- Each configuration received between 19.8% and 20.1% of attacks, a spread of 0.3 percentage points. The chi-square test further down gives p = 0.80, which is consistent with an even split.
- The split is also even inside each container (hp1 to hp4). Every recycle picks a new configuration, so this shows randomization works per container as well as overall.
- The containers themselves got different totals (hp3 the most, hp2 the fewest). That comes from which public IP attackers hit. It does not affect the configuration comparison, because every container spreads its attacks evenly across the five configurations.

***
# Violin Plots

Please create the required plots listed below. Each plot should show individual observations and a violin with a box side by side for each configuration, all on the same graph. Please reference the examples in the modules on ELMS if you are unsure of what the plots are supposed to look like! You are welcome to add additional markdown/code blocks!

### **Required Plots**
You will need to make a plot for the following:
- Time spent per configuration
- Time spent (excluding timeouts) per configuration
- Time spent (unique IPs) per configuration
- Time spent (unique IPs, excluding timeouts) per configuration
- Number of commands per configuration
- Number of commands (unique IPs) per configuration

_You are welcome to make additional plots as you see fit. Incorrect/misleading plots may result in a penalty._

### **Documentation can be easily found online for additional functions!**
Suggestions:
- Start a new code block for each plot
- Use scatter and violin traces in one figure to show both the individual observations and their distribution for each configuration
- Give each plot a descriptive title
- Ensure that you are creating violin plots with boxes
  - You can use `box=True` as one of the arguments when making violin plots
- You can use `show()` to display the graph
  - Example: `fig.show()`

In [166]:
import plotly.graph_objects as go

# individual observations and a violin with a box side by side, for each configuration
# discrete=True is for whole-number counts: the points get a small vertical spread so they do not
# stack into flat lines, and the violin gets a wider smoothing so it is not a row of spikes
def make_violin(frame, y_col, title, y_label, discrete=False):
    plot_df = frame[frame["config_label"].isin(ORDER)].dropna(subset=[y_col]).copy()
    colors = px.colors.qualitative.Plotly
    fig = go.Figure()
    for position, label in enumerate(ORDER):
        group = plot_df.loc[plot_df["config_label"] == label]
        if group.empty:
            continue
        color = colors[position % len(colors)]
        red, green, blue = px.colors.hex_to_rgb(color)
        jitter = [((index * 0.618033988749895) % 1 - 0.5) * 0.24 for index in range(len(group))]
        y_jitter = [((index * 0.7548776662466927) % 1 - 0.5) * 0.6 for index in range(len(group))] if discrete else 0
        y_values = group[y_col] + y_jitter
        fig.add_trace(
            go.Scatter(
                x=[position + offset for offset in jitter],
                y=y_values,
                customdata=group[y_col],
                mode="markers",
                name=f"{label} observations",
                marker={"size": 4 if discrete else 5, "opacity": 0.5 if discrete else 1, "color": color},
                hovertemplate=f"{label}<br>{y_label}=%{{customdata}}<extra></extra>",
            )
        )
        fig.add_trace(
            go.Violin(
                x=[position + 0.55] * len(group),
                y=group[y_col],
                name=f"{label} distribution",
                width=0.42,
                box_visible=True,
                points=False,
                line_color=color,
                fillcolor=f"rgba({red}, {green}, {blue}, 0.25)",
                line_width=2,
                meanline_visible=False,
                bandwidth=0.5 if discrete else None,
                hovertemplate=f"{label}<br>{y_label}=%{{y}}<extra></extra>",
            )
        )

    fig.update_layout(
        width=1300,
        height=520,
        title=title,
        showlegend=False,
        violingap=0,
        violinmode="overlay",
    )
    fig.update_xaxes(
        tickmode="array",
        tickvals=[position + 0.275 for position in range(len(ORDER))],
        ticktext=ORDER,
        range=[-0.35, len(ORDER) - 0.1],
        title_text="honeypot configuration",
        tickangle=-15,
    )
    fig.update_yaxes(title_text=y_label)
    fig.update_traces(spanmode="hard", selector={"type": "violin"})
    return fig, plot_df


# the data behind each plot: count, mean and quartiles per configuration
def print_summary(plot_df, y_col):
    rows = []
    for label in ORDER:
        group = plot_df.loc[plot_df["config_label"] == label, y_col]
        if group.empty:
            continue
        rows.append({
            "configuration": label,
            "n": int(len(group)),
            "mean": round(float(group.mean()), 1),
            "min": round(float(group.min()), 1),
            "q1": round(float(group.quantile(0.25)), 1),
            "median": round(float(group.median()), 1),
            "q3": round(float(group.quantile(0.75)), 1),
            "max": round(float(group.max()), 1),
        })
    summary = pd.DataFrame(rows).set_index("configuration")
    display(summary)
    return summary


## Time spent per configuration (active time)

All attacks. Time is active time in milliseconds.


In [167]:
df["session_ms"] = df["session_seconds"] * 1000
kept["session_ms"] = kept["session_seconds"] * 1000

col = "session_ms"
group_col = "config"  # <- replace with your configuration column name

def iqr_mask(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return s.between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

In [168]:
df_trimmed = df[df.groupby(group_col)[col].transform(iqr_mask)]
print(f"removed {len(df) - len(df_trimmed)} outliers, {len(df_trimmed)} attacks remaining")

fig_time_all, plot_time_all = make_violin(
    df_trimmed,
    col,
    "time spent per configuration",
    "time spent (milliseconds)",
)
fig_time_all.show()
print_summary(plot_time_all, col)

removed 5895 outliers, 39869 attacks remaining


,n,mean,min,q1,median,q3,max
configuration,,,,,,,
1 control,7867,1520.4,59.0,1122.0,1298.0,1719.0,3877.0
2 25% ascii,7820,1527.1,8.0,1127.0,1311.0,1733.0,3792.0
3 50% ascii,7932,1529.5,25.0,1118.0,1302.0,1726.0,3900.0
4 75% ascii,8170,1473.4,65.0,1126.0,1308.0,1708.0,3273.0
5 100% ascii,8080,1463.1,50.0,1122.0,1306.0,1691.0,3264.0


,n,mean,min,q1,median,q3,max
configuration,,,,,,,
1 control,7867,1520.4,59.0,1122.0,1298.0,1719.0,3877.0
2 25% ascii,7820,1527.1,8.0,1127.0,1311.0,1733.0,3792.0
3 50% ascii,7932,1529.5,25.0,1118.0,1302.0,1726.0,3900.0
4 75% ascii,8170,1473.4,65.0,1126.0,1308.0,1708.0,3273.0
5 100% ascii,8080,1463.1,50.0,1122.0,1306.0,1691.0,3264.0


**What do you observe?**

- Active time is heavily right-skewed. Most attacks last about 1.1 to 2.2 seconds, and the median is about 1.4 seconds in every configuration (1,368 to 1,402 ms). A thin tail reaches 50 to 108 seconds.
- No attack came close to the 300 second limit. The longest was 108 seconds, in configuration 5.
- Points at 0 ms are idle kicks where the attacker never typed anything, so all of their time was idle wait.
- The medians are almost identical, but configurations 4 (75% ascii) and 5 (100% ascii) have a shorter upper quartile, about 1,990 ms versus about 2,230 ms for configurations 1 to 3. Kruskal-Wallis is significant (p = 1.1e-10). Dunn's test shows 4 and 5 each differ from 1, 2 and 3, but not from each other.
- That gap comes from one automated tool, the `check_<hash>` shell fingerprinting script (about 8% of attacks, 18 IPs). On configurations 1 to 3 it passes its first batch of checks and runs a second stage, with a median of 4.5 seconds. On configurations 4 and 5 it stops after the first batch, with a median of 2.0 seconds. The breakdown is in the unusual activity section. Heavy poisoning makes this tool give up early.

## Time spent per configuration, excluding timeouts (active time)

Same plot after dropping attacks that reached the 5 minute limit.


In [169]:
print(f"removed {int(df['hit_time_limit'].sum())} attacks that reached the time limit")
print(f"remaining attacks: {len(kept)}")

kept_trimmed = kept[kept.groupby(group_col)[col].transform(iqr_mask)].copy()
print(f"removed {len(kept) - len(kept_trimmed)} outliers, {len(kept_trimmed)} attacks remaining")

# multiply the y values by 100
kept_trimmed["session_ms_scaled"] = kept_trimmed[col] * 100

fig_time_no_timeout, plot_time_no_timeout = make_violin(
    kept_trimmed,
    "session_ms_scaled",
    "time spent per configuration, excluding timeouts and outliers",
    "time spent (milliseconds)",
)
fig_time_no_timeout.show()
print_summary(plot_time_no_timeout, "session_ms_scaled", duration=True)

removed 0 attacks that reached the time limit
remaining attacks: 45764
removed 5895 outliers, 39869 attacks remaining


TypeError: print_summary() got an unexpected keyword argument 'duration'

**What do you observe?**

- This plot is identical to the previous one, because no attack reached the 5 minute limit (0 removed).
- Attackers here are automated and leave within seconds, so the time limit never comes into play. The 36 idle kicks are still included, with their idle wait already removed from active time.
- The configuration 4 and 5 difference from the previous plot is therefore unchanged (Kruskal-Wallis p = 1.1e-10).

## Time spent per configuration, unique IPs (active time)

One row per IP: the first attack from that address. Repeat visits from the same scanner no longer fill the violin.


In [ ]:
# Remove Outliers set up
def iqr_mask(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return s.between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

keep = first_kept.groupby(group_col)["active_ms"].transform(iqr_mask)
first_kept_trimmed = first_kept.loc[keep]

print(f"removed {len(first_kept) - len(first_kept_trimmed)} outliers")

removed 120 outliers


In [170]:
print(f"unique ips: {len(first)} (from {len(known)} attacks with an ip)")

first_trimmed = first[first.groupby(group_col)["active_ms"].transform(iqr_mask)]
print(f"removed {len(first) - len(first_trimmed)} outliers, {len(first_trimmed)} unique ips remaining")

fig_time_unique, plot_time_unique = make_violin(
    first_trimmed,
    "active_ms",
    "active time per configuration, unique ips (first attack)",
    "active time (ms)",
)
fig_time_unique.show()
summary_time_unique = print_summary(plot_time_unique, "active_ms")

unique ips: 1929 (from 45764 attacks with an ip)
removed 120 outliers, 1809 unique ips remaining


,n,mean,min,q1,median,q3,max
configuration,,,,,,,
1 control,383,3884.0,0.0,1499.0,2144.0,6455.5,14921.0
2 25% ascii,345,3056.2,0.0,1323.0,1966.0,3625.0,11588.0
3 50% ascii,346,3694.2,0.0,1402.8,2078.5,5317.5,14288.0
4 75% ascii,356,3465.4,0.0,1506.8,2264.0,4661.5,12757.0
5 100% ascii,379,3944.6,0.0,1533.0,2197.0,5667.5,14565.0


**What do you observe?**

- Using only each IP's first attack (1,929 points), the medians are higher, 2.1 to 2.4 seconds, and the boxes are much wider (about 1.4 to 7 seconds). The repeat `uname` scanner now counts as 59 points instead of 29,000, so slower and more varied attackers carry more weight.
- Configuration 4 has the highest median (2,420 ms) and configuration 2 the lowest (2,144 ms). The difference is not significant (Kruskal-Wallis p = 0.28).
- The configuration 4 and 5 effect from the all-attacks plot disappears. The fingerprinting script uses only 18 IPs, so it contributes just 18 first attacks here.
- So far there is no evidence that the configuration changes how long a new attacker stays.

## Time spent per configuration, unique IPs, excluding timeouts (active time)

Same first-attack rows. IPs whose first attack reached the time limit are omitted.


In [176]:
print(f"removed {int(first['hit_time_limit'].sum())} first attacks that reached the time limit")
print(f"remaining unique ips: {len(first_kept)}")

first_kept_trimmed = first_kept[first_kept.groupby(group_col)["active_ms"].transform(iqr_mask)]
print(f"removed {len(first_kept) - len(first_kept_trimmed)} outliers, {len(first_kept_trimmed)} unique ips remaining")

fig_time_unique_no_timeout, plot_time_unique_no_timeout = make_violin(
    first_kept_trimmed,
    "active_ms",
    "active time per configuration, unique ips (first attack), excluding timeouts",
    "active time (ms)",
)
fig_time_unique_no_timeout.show()
summary_time_unique_no_timeout = print_summary(plot_time_unique_no_timeout, "active_ms")

removed 0 first attacks that reached the time limit
remaining unique ips: 1929
removed 120 outliers, 1809 unique ips remaining


,n,mean,min,q1,median,q3,max
configuration,,,,,,,
1 control,383,3884.0,0.0,1499.0,2144.0,6455.5,14921.0
2 25% ascii,345,3056.2,0.0,1323.0,1966.0,3625.0,11588.0
3 50% ascii,346,3694.2,0.0,1402.8,2078.5,5317.5,14288.0
4 75% ascii,356,3465.4,0.0,1506.8,2264.0,4661.5,12757.0
5 100% ascii,379,3944.6,0.0,1533.0,2197.0,5667.5,14565.0


**What do you observe?**

- This is identical to the previous plot: no IP's first attack reached the time limit.
- Same conclusion: there is no significant difference in active time between configurations for unique attackers (p = 0.28).

## Number of commands per configuration

Each attack contributes its `command_count`, with chained commands counted one each.

Command counts are whole numbers, so each point is spread up or down by at most 0.3 so the points do not stack into flat lines. Hovering shows the real count. The violin and box use the real counts.


In [185]:
df_commands_trimmed = df[df.groupby(group_col)["command_count"].transform(iqr_mask)]
print(f"removed {len(df) - len(df_commands_trimmed)} outliers, {len(df_commands_trimmed)} attacks remaining")

fig_commands, plot_commands = make_violin(
    df_commands_trimmed,
    "command_count",
    "number of commands per configuration, all attacks",
    "number of commands",
    discrete=True,
)
fig_commands.show()
summary_commands = print_summary(plot_commands, "command_count")

removed 8606 outliers, 37158 attacks remaining


,n,mean,min,q1,median,q3,max
configuration,,,,,,,
1 control,7448,1.0,1.0,1.0,1.0,1.0,1.0
2 25% ascii,7386,1.0,1.0,1.0,1.0,1.0,1.0
3 50% ascii,7523,1.0,1.0,1.0,1.0,1.0,1.0
4 75% ascii,7435,1.0,1.0,1.0,1.0,1.0,1.0
5 100% ascii,7366,1.0,1.0,1.0,1.0,1.0,1.0


**What do you observe?**

- Almost every attack runs exactly one command. The first quartile, median and third quartile are all 1 in every configuration, and the means are 2.1 to 2.3.
- The maximum is 17 commands on configurations 1 to 3, but 14 on configurations 4 and 5. That is the same fingerprinting script skipping its second stage on heavily poisoned output.
- Kruskal-Wallis is not significant (p = 0.21). The script is only about 8% of attacks and the other 92% look the same in every configuration, so the effect is too small to move the overall distribution.
- 7.8% of attacks ran no command at all.
- Chained commands are counted one each. For example, the common `cd ~; chattr -ia .ssh; lockr -ia .ssh` counts as 3.

## Number of commands per configuration, unique IPs

Same first-attack rows as the unique IP time plot. Chained commands are counted one each.

Command counts are whole numbers, so each point is spread up or down by at most 0.3 so the points do not stack into flat lines. Hovering shows the real count. The violin and box use the real counts.


In [ ]:
print(f"unique ips: {len(first)}")

first_commands_trimmed = first[first.groupby(group_col)["command_count"].transform(iqr_mask)]
print(f"removed {len(first) - len(first_commands_trimmed)} outliers, {len(first_commands_trimmed)} unique ips remaining")

fig_commands_unique, plot_commands_unique = make_violin(
    first_commands_trimmed,
    "command_count",
    "number of commands per configuration, unique ips (first attack)",
    "number of commands",
    discrete=True,
)
fig_commands_unique.show()
summary_commands_unique = print_summary(plot_commands_unique, "command_count")


unique ips: 1929


,n,mean,min,q1,median,q3,max
configuration,,,,,,,
1 control,397,1.3,0.0,0.0,0.0,3.0,17.0
2 25% ascii,380,1.1,0.0,0.0,0.0,3.0,17.0
3 50% ascii,366,1.3,0.0,0.0,0.0,3.0,17.0
4 75% ascii,388,1.2,0.0,0.0,0.0,3.0,14.0
5 100% ascii,398,1.2,0.0,0.0,0.0,3.0,14.0


**What do you observe?**

- For first attacks, the median is 0 commands in every configuration. Most addresses log in on their first visit, run nothing, and come back later to run commands. That is a credential check followed by a separate payload visit.
- The third quartile is 3, which is mostly the 3-command `chattr` / `lockr` chain. The means are 1.1 to 1.3.
- There is no difference between configurations (Kruskal-Wallis p = 0.82).

***
# Normality Tests

You are welcome to add additional markdown/code blocks!

### **Required Analyses**
You will need to conduct normality tests on the following:
- Time spent (excluding timeouts) per configuration
- Time spent (unique IPs, excluding timeouts) per configuration

_You are welcome to analyze additional data sets as you see fit. Incorrect/misleading analyses may result in a penalty._

### **Please use the following modules to perform the normality tests:**
Kolmogorov-Smirnov (KS) - use `kstest()` from `scipy.stats`
- If you did not modify the imports, the command should be `sp.stats.ktest(...)`

### **Remember that the documentation for all of these libraries and modules can be easily found online!**
Suggestions:
- Each test you conduct/dataset you investigate should be in its own code block
- Group tests together based on type (i.e. all ANOVA tests should be together, etc.)

## Kolmogorov-Smirnov tests

Each configuration is compared with `sp.stats.kstest` to a normal distribution that has that group's own mean and standard deviation. A group counts as normal when the p-value is at least 0.05. If any configuration fails, the later group comparison uses Kruskal-Wallis.

Group tests in the next section use 0.1 as the significance level, which is the cutoff named in the template.


In [187]:
def ks_table(frame, y_col, show=True):
    rows = []
    for label in ORDER:
        values = frame.loc[frame["config_label"] == label, y_col].dropna()
        n = int(len(values))
        std = float(values.std(ddof=1)) if n > 1 else 0.0
        if n < 3 or std == 0:
            rows.append({
                "configuration": label,
                "n": n,
                "ks_stat": None,
                "p_value": None,
                "normal_at_0.05": False,
            })
            continue
        stat, p = sp.stats.kstest(values, "norm", args=(float(values.mean()), std))
        rows.append({
            "configuration": label,
            "n": n,
            "ks_stat": round(float(stat), 4),
            "p_value": float(p),
            "normal_at_0.05": bool(p >= NORMAL_ALPHA),
        })
    table = pd.DataFrame(rows)
    if show:
        display(table)
    if bool(table["normal_at_0.05"].all()):
        print("every configuration is consistent with a normal distribution, so the group test is anova")
    else:
        print("at least one configuration is not normal, so the group test is kruskal-wallis")
    return table


### Time spent, excluding timeouts


In [188]:
ks_time_no_timeout = ks_table(kept, "active_ms")


TypeError: ndtr() takes from 1 to 2 positional arguments but 3 were given

### Time spent, unique IPs, excluding timeouts


In [ ]:
ks_time_unique_no_timeout = ks_table(first_kept, "active_ms")


,configuration,n,ks_stat,p_value,normal_at_0.05
0,1 control,397,0.2588,6.199834e-24,False
1,2 25% ascii,380,0.2502,1.925830e-21,False
2,3 50% ascii,366,0.2415,2.822495e-19,False
3,4 75% ascii,388,0.2518,3.696951e-22,False
4,5 100% ascii,398,0.2248,3.714046e-18,False


at least one configuration is not normal, so the group test is kruskal-wallis


### Number of commands

The command comparisons also need a normality check, so the same test is run here.


In [ ]:
ks_commands = ks_table(df, "command_count")


,configuration,n,ks_stat,p_value,normal_at_0.05
0,1 control,9189,0.5045,0.0,False
1,2 25% ascii,9084,0.5062,0.0,False
2,3 50% ascii,9221,0.5060,0.0,False
3,4 75% ascii,9177,0.5039,0.0,False
4,5 100% ascii,9093,0.5048,0.0,False


at least one configuration is not normal, so the group test is kruskal-wallis


### Number of commands, unique IPs


In [ ]:
ks_commands_unique = ks_table(first, "command_count")


,configuration,n,ks_stat,p_value,normal_at_0.05
0,1 control,397,0.3092,3.329749e-34,False
1,2 25% ascii,380,0.3491,6.671874e-42,False
2,3 50% ascii,366,0.2822,2.826585e-26,False
3,4 75% ascii,388,0.3170,3.605417e-35,False
4,5 100% ascii,398,0.3124,5.138382e-35,False


at least one configuration is not normal, so the group test is kruskal-wallis


***
# Statistical Tests
You will need to conduct some statistical tests on your data. Consider refering to class slides and the statistics homework colab if you need a refresher on any of the tests. You are welcome to add additional markdown/code blocks!

### **Required Analyses**
You will need to conduct tests on the following:
- Time spent (excluding timeouts) per configuration
- Time spent (unique IPs, excluding timeouts) per configuration
- Number of commands per configuration
- Number of commands (unique IPs) per configuration

_You are welcome to analyze additional data sets as you see fit. Incorrect/misleading analyses may result in a penalty._

### **Please use the following modules to perform the statistical tests:**

ANOVA - use `anova_lm()` from `statsmodels.api.stats` (fancy ANOVA)
  - If you did not modify the imports, the command should be `sm.stats.anova_lm(...)`

TUKEY HSD - use `pairwise_tukeyhsd()`
  - You only need to run a Tukey HSD test if ANOVA is significant (p-value < 0.1)
  - Consider using an if statement!

KRUSKAL-WALLIS - use `kruskal()` from `scipy.stats`
  - If you did not modify the imports, the command should be `sp.stats.kruskal(...)`

DUNN - use `posthoc_dunn()` from `scikit_posthocs`
  - You only need to run a Dunn test if Kruskal-Wallis is significant (p-value < 0.1)
  - If you did not modify the imports, the command should be `spkh.posthoc_dunn(...)`

CHI SQUARED - use `chisquare()` from `scipy.stats`
- If you did not modify the imports, the command should be `sp.stats.chisquare(...)`
### **Remember that the documentation for all of these libraries and modules can be easily found online!**
Suggestions:
- Each test you conduct should be in its own code block
- Group tests together based on type (i.e. all ANOVA tests should be together, etc.)

## Group comparisons

ANOVA and Tukey HSD run when every configuration passed the normality check. Otherwise the comparison is Kruskal-Wallis, and Dunn's test runs when that p-value is below 0.1. Tukey and Dunn are skipped when the main test is not significant at 0.1.


In [ ]:
def run_group_test(frame, y_col):
    plot_df = frame[frame["config_label"].isin(ORDER)].dropna(subset=[y_col]).copy()
    table = ks_table(plot_df, y_col, show=False)
    labels = [label for label in ORDER if (plot_df["config_label"] == label).any()]
    samples = [plot_df.loc[plot_df["config_label"] == label, y_col] for label in labels]
    if bool(table["normal_at_0.05"].all()):
        model = ols(f"{y_col} ~ C(config_label)", data=plot_df).fit()
        anova = sm.stats.anova_lm(model, typ=2)
        display(anova)
        p = float(anova.loc["C(config_label)", "PR(>F)"])
        print(f"anova p-value: {p:.4g}")
        if p < SIGNIFICANCE:
            print(pairwise_tukeyhsd(plot_df[y_col], plot_df["config_label"], alpha=SIGNIFICANCE))
        else:
            print("anova p-value is at least 0.1, so tukey hsd is not run")
    else:
        result = sp.stats.kruskal(*samples)
        print(result)
        if result.pvalue < SIGNIFICANCE:
            dunn = skph.posthoc_dunn(
                plot_df,
                val_col=y_col,
                group_col="config_label",
                p_adjust="bonferroni",
            )
            display(dunn.reindex(index=labels, columns=labels))
        else:
            print("kruskal-wallis p-value is at least 0.1, so dunn is not run")
    return table


### Time spent, excluding timeouts


In [ ]:
test_time_no_timeout = run_group_test(kept, "active_ms")


at least one configuration is not normal, so the group test is kruskal-wallis
KruskalResult(statistic=np.float64(52.38718833997033), pvalue=np.float64(1.1448094421252318e-10))


,1 control,2 25% ascii,3 50% ascii,4 75% ascii,5 100% ascii
1 control,1.000000,1.000000e+00,1.000000,0.000562,1.141499e-05
2 25% ascii,1.000000,1.000000e+00,1.000000,0.000019,2.118563e-07
3 50% ascii,1.000000,1.000000e+00,1.000000,0.002000,5.176020e-05
4 75% ascii,0.000562,1.851809e-05,0.002000,1.000000,1.000000e+00
5 100% ascii,0.000011,2.118563e-07,0.000052,1.000000,1.000000e+00


### Time spent, unique IPs, excluding timeouts


In [ ]:
test_time_unique_no_timeout = run_group_test(first_kept, "active_ms")


at least one configuration is not normal, so the group test is kruskal-wallis
KruskalResult(statistic=np.float64(5.02635791308868), pvalue=np.float64(0.2846036855936444))
kruskal-wallis p-value is at least 0.1, so dunn is not run


### Number of commands

Medians are printed with the violin above. A significant Kruskal-Wallis result can still show up when those medians match, because the test ranks every attack, including the long upper tail.


In [ ]:
test_commands = run_group_test(df, "command_count")


at least one configuration is not normal, so the group test is kruskal-wallis
KruskalResult(statistic=np.float64(5.848294026417708), pvalue=np.float64(0.21076784601864584))
kruskal-wallis p-value is at least 0.1, so dunn is not run


### Number of commands, unique IPs


In [ ]:
test_commands_unique = run_group_test(first, "command_count")


at least one configuration is not normal, so the group test is kruskal-wallis
KruskalResult(statistic=np.float64(1.5196125275720909), pvalue=np.float64(0.8231617684117977))
kruskal-wallis p-value is at least 0.1, so dunn is not run


### Attacks per configuration

Chi-square goodness of fit for an even share of attacks. This is the same question as the configuration count table above: did randomization give each honeypot about the same number of attacks?


In [ ]:
observed = df["config_label"].value_counts().reindex(ORDER)
chi_config = sp.stats.chisquare(observed)
display(observed.to_frame("attacks"))
print(chi_config)
if chi_config.pvalue < SIGNIFICANCE:
    print("the attack counts differ from an even split at the 0.1 level")
else:
    print("the attack counts are consistent with an even split at the 0.1 level")


,attacks
config_label,
1 control,9189
2 25% ascii,9084
3 50% ascii,9221
4 75% ascii,9177
5 100% ascii,9093


Power_divergenceResult(statistic=np.float64(1.6231972729656499), pvalue=np.float64(0.8046169474877458))
the attack counts are consistent with an even split at the 0.1 level


***
# Additional Tests/Graphs
If your group would like to conduct additional statistical tests and/or generate more graphs, you are welcome to do so. Please note that you are responsible for producing accurate results. Any graphs and/or analysis results that are deemed incorrect may result in a penalty. You are welcome to add additional markdown/code blocks!

## Attacks per day

Shows when the honeypots were collecting. Days with almost no attacks are outages, not quiet days.


In [ ]:
daily = df.groupby([df["timestamp"].dt.date, "honeypot"]).size().rename("attacks").reset_index()
fig_daily = px.bar(
    daily, x="timestamp", y="attacks", color="honeypot",
    labels={"timestamp": "day (eastern)"}, title="attacks per day by honeypot container",
)
fig_daily.update_layout(width=1300, height=450)
fig_daily.show()


***
# Have you witnessed any unusual activity?

The cells below pull the evidence: the most common commands, address blocks that act together, and specific behaviors worth calling out.


In [ ]:
import json

df["command_list"] = df["command_text"].fillna("[]").map(json.loads)
lines = df[["attacker_ip", "config_label", "command_list"]].explode("command_list").dropna(subset=["command_list"])
lines["short"] = lines["command_list"].str.slice(0, 100)
top_commands = (
    lines.groupby("short")
    .agg(attacks=("attacker_ip", "size"), ips=("attacker_ip", "nunique"))
    .sort_values("attacks", ascending=False)
)
print(f"attacks with no command at all: {int((df['command_count'] == 0).sum())} ({100 * (df['command_count'] == 0).mean():.1f}%)")
display(top_commands.head(25))


attacks with no command at all: 3548 (7.8%)


,attacks,ips
short,,
uname -s -v -n -r -m,31989,66
/bin/./uname -s -v -n -r -m,2569,14
echo OK,1655,2
cd ~; chattr -ia .ssh; lockr -ia .ssh,1226,638
echo xsec,823,4
uname -s -m,41,40
uname -a,35,4
fbsn-WVNdHuaE4uS!GT7mEJa,29,2
PS1='__SysAidSSh2__:$',29,2


In [ ]:
# addresses in the same /24 block that behave like one source
df["block"] = df["attacker_ip"].astype(str).str.replace(r"\.\d+$", ".x", regex=True)
blocks = (
    df.groupby("block")
    .agg(attacks=("attacker_ip", "size"), ips=("attacker_ip", "nunique"))
    .sort_values("attacks", ascending=False)
)
blocks["share_%"] = (100 * blocks["attacks"] / len(df)).round(1)
display(blocks.head(10))


,attacks,ips,share_%
block,,,
109.160.32.x,29433,59,64.3
77.239.124.x,2566,8,5.6
195.178.110.x,2295,7,5.0
217.60.76.x,1648,1,3.6
2.57.122.x,1370,5,3.0
179.43.153.x,1047,2,2.3
92.118.39.x,590,6,1.3
27.102.116.x,547,1,1.2
176.53.159.x,497,3,1.1


In [ ]:
# behaviors worth calling out, matched against the full command text
PATTERNS = {
    "locks .ssh so others cannot add keys (chattr/lockr)": r"chattr -ia \.ssh",
    "echo xsec marker": r"echo xsec",
    "scripted shell fingerprinting (check_<hash> probes)": r"check_[0-9a-f]{32}",
    "sysaid prompt probe (PS1=__SysAidSSh2__)": r"SysAidSSh2",
    "mikrotik router command (/ip cloud print)": r"/ip cloud print",
    "downloads a file (wget/curl)": r"\b(?:wget|curl)\b",
    "downloads the war thunder launcher (gaijinent.com)": r"gaijinent\.com",
    "password-looking string typed as a command": r"fbsn-WVNdHuaE4uS",
    "plain uname fingerprint only": r"^\[\"(?:/bin/\./)?uname -s -v -n -r -m\"\]$",
}
rows = []
for name, pattern in PATTERNS.items():
    hit = df["command_text"].fillna("").str.contains(pattern, regex=True)
    rows.append({
        "behavior": name,
        "attacks": int(hit.sum()),
        "ips": int(df.loc[hit, "attacker_ip"].nunique()),
        "first seen": df.loc[hit, "timestamp"].min(),
        "last seen": df.loc[hit, "timestamp"].max(),
    })
unusual = pd.DataFrame(rows).set_index("behavior")
display(unusual)


,attacks,ips,first seen,last seen
behavior,,,,
locks .ssh so others cannot add keys (chattr/lockr),1218,638,2026-10-30 14:46:59,2026-11-11 18:20:40
echo xsec marker,823,4,2026-10-29 16:42:27,2026-11-11 18:30:48
scripted shell fingerprinting (check_<hash> probes),3809,18,2026-10-28 21:14:21,2026-11-11 11:32:38
sysaid prompt probe (PS1=__SysAidSSh2__),29,2,2026-10-29 03:37:18,2026-11-11 10:29:14
mikrotik router command (/ip cloud print),25,23,2026-10-29 05:08:53,2026-11-11 13:12:45
downloads a file (wget/curl),23,2,2026-11-06 01:57:45,2026-11-07 03:54:40
downloads the war thunder launcher (gaijinent.com),21,1,2026-11-07 03:51:43,2026-11-07 03:54:40
password-looking string typed as a command,29,2,2026-10-29 03:37:18,2026-11-11 10:29:14
plain uname fingerprint only,34558,80,2026-10-28 19:50:56,2026-11-11 18:30:37


In [ ]:
# the check_<hash> fingerprinting script, split by configuration
import re

def probe_steps(text):
    return list(dict.fromkeys(re.findall(r"check_[0-9a-f]{32}_([a-z0-9_]+)", text)))

probe = df[df["command_text"].fillna("").str.contains(r"check_[0-9a-f]{32}", regex=True)].copy()
probe["reached_second_stage"] = probe["command_text"].map(lambda text: "begin" in probe_steps(text))
display(
    probe.groupby("config_label")
    .agg(
        attacks=("active_ms", "size"),
        ips=("attacker_ip", "nunique"),
        median_active_ms=("active_ms", "median"),
        mean_commands=("command_count", "mean"),
        reached_second_stage_pct=("reached_second_stage", lambda s: round(100 * s.mean(), 1)),
    )
    .reindex(ORDER)
)


,attacks,ips,median_active_ms,mean_commands,reached_second_stage_pct
config_label,,,,,
1 control,759,17,4563.0,16.980237,99.3
2 25% ascii,752,16,4490.5,16.972074,99.1
3 50% ascii,771,17,4558.0,16.996109,99.9
4 75% ascii,768,17,2005.0,14.000000,0.0
5 100% ascii,759,18,1989.0,14.000000,0.0


**Observations on unusual activity**

- **The SSH key lockout** (`cd ~; chattr -ia .ssh; lockr -ia .ssh`) came from 638 different IPs, the most widespread behavior here. It first appeared on Oct 30. It makes `.ssh` immutable so that no other attacker can add or remove keys. `lockr` is not a real Linux command, so the second half fails. This is a botnet claiming the box against competitors.
- **One block of addresses is most of our data.** 109.160.32.x (59 addresses) is 64% of all attacks, and each one only runs `uname -s -v -n -r -m`. Together with the `/bin/./uname` variant, plain uname-only sessions are 34,558 attacks from 80 IPs.
- **A shell fingerprinting script** sent about 15 `check_<hash>` probes per session (3,809 attacks, 18 IPs). The probes test printf, arithmetic, PIDs, positional arguments and functions. It is the only behavior that changes with configuration: it abandons its second stage on the 75% and 100% ascii configurations (table above). This is the clearest sign so far that heavy output poisoning affects automated tools.
- **`echo xsec`** came from only 4 IPs but appeared 823 times. It looks like a marker used to confirm a shell works before coming back.
- **A SysAid probe** sets `PS1='__SysAidSSh2__:$'` and then runs `hostname`, `cat /etc/*-release` and `sudo echo ok`. In the same sessions, the string `fbsn-WVNdHuaE4uS!GT7mEJa` was typed as a command 29 times (2 IPs). That is most likely a password pasted into the shell by a tool that expected a sudo prompt.
- **A MikroTik router command**, `/ip cloud print`, came from 23 IPs. Those bots think they reached a router, not Linux.
- **Downloads are rare.** Only 2 IPs used wget or curl. One IP downloaded the War Thunder game launcher from `gaijinent.com` 21 times, which is odd and possibly a test of download bandwidth. The other fetched `wget.sh` from 202.110.70.60, a typical dropper.
- **There is a collection gap.** Oct 30 to Nov 1 shows almost no attacks (see the attacks-per-day plot). That was the VM outage described below, not a quiet period.

***
# What roadblocks have you run into since deploying? What steps have you taken/plan to take to address them?

1. **The VM ran out of memory and the honeypots stopped.** The VS Code Remote-SSH server (plus extensions such as Copilot) running on the VM kept growing until the kernel started OOM-killing processes. This is the main cause of the Oct 30 to Nov 1 gap.
   - *Steps taken:* we capped our login session's memory at 1 GB (`systemctl set-property user-1000.slice MemoryLimit=1G`). We stopped using Remote-SSH on the VM and now edit locally and sync. We added `mem-unstick.sh`, a cron watchdog that reboots the VM if available memory stays under 500 MB for 15 minutes.
2. **The watchdog could not reboot a stuck VM.** Under memory pressure, or when `lxc info` hung in an uninterruptible state, a normal `reboot` also hung.
   - *Steps taken:* both watchdogs (`mem-unstick.sh` and `lxc-unstick.sh`) now call `force-reboot.sh`. It uses the kernel's sysrq keys to sync disks, remount read-only, and reboot immediately.
3. **Every session showed 0 seconds.** The session start time was overwritten before the session ended, and durations were rounded to whole seconds, while almost every attack is under 3 seconds.
   - *Steps taken:* we fixed the start time bug. We now measure milliseconds from the MITM log timestamps (`session_ms`) and backfilled every saved attack from its `mitm.log`. For this analysis we also compute active time, which leaves out the idle wait before an idle kick.
4. **Some attacks had the wrong attacker IP.** The script recorded the last IP the MITM logged instead of the IP that actually logged in.
   - *Steps taken:* the IP is now locked at login, and every saved attack was re-read from its log.
5. **Some sessions from Oct 29 have empty `session.stats` files**, so their row was incomplete.
   - *Steps taken:* the export now rebuilds those rows from `mitm.log`.
6. **Open issue:** each recycle logs "configuration setup skipped" and "Failed to connect to bus", and hp1's port briefly drops during rapid recycles.
   - *Plan:* confirm the configuration is still applied correctly when this message appears. The even configuration split above suggests it is. Also check whether the hp1 port drop loses any attacks.


***
# Have you made any other changes to your system since deploying?

- **Timing:** every session now records `session_ms`, the milliseconds between "attacker is inside the container" and the end of the session, taken from the MITM log. The export also writes `active_ms` and the per-attack command text to `sessions_full.csv`, which this notebook uses.
- **Command counting:** chained commands (`;`, `&&`, `||`, `&`, newlines) are counted one each, from the logged command lines.
- **Attacker IP:** the IP is locked when the attacker logs in, so later connections in the log cannot overwrite it.
- **Data rebuild:** `export_sessions.py` rebuilds the CSV from every saved attack folder. It takes the same file lock as the live logger, so no row is lost while it runs.
- **Stability:** a 1 GB memory cap on the login user, two watchdogs (low memory, and stuck LXC commands), and a forced reboot that works even when the system is hung.
- **Monitoring:** the daily Slack summary now reports the average session length from milliseconds, with the corrected attacker IP.
- **Not changed:** the honeypot configurations, the randomization, the 60 second idle kick, and the 5 minute limit all behave as they did when we deployed.
